# Sweet Sorghum – Land Use Masks + Area Summary (ArcPy)

This notebook creates three raster masks from a CDL/CropScape-style land-use raster:

1. **Current agriculture** (`CurrentAg_R`)  
2. **Current ag + sorghum-available land** (`CurrentAg_and_SorghumAvailable_R`)  
3. **Sorghum-available land that is not currently agriculture** (`SorghumAvailable_NotCurrentAg_R`)  

It also calculates area totals (m², hectares, acres) and the percent of the study area currently in agriculture.


## 1) Setup (edit paths / raster name here)

In [ ]:
from pathlib import Path
import arcpy
import os
import datetime as dt
import pandas as pd

from arcpy.sa import Raster, Con, IsNull

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# -------------------------
# INPUTS
# -------------------------
# Project geodatabase
GDB = r"C:\School\Fall 25\RA Sweet Sorghum\Project\SweetSorghum\SweetSorghum.gdb"

# Land-use / crop raster (CDL/CropScape-style) stored in the GDB
LANDUSE_RASTER_NAME = "MN_2024_10m_Crop_Clip"

# -------------------------
# OUTPUT NAMES (in the same GDB)
# -------------------------
CURRENT_AG_NAME        = "CurrentAg_R"
SORGHUM_AVAILABLE_NAME = "CurrentAg_and_SorghumAvailable_R"
AVAILABLE_NOT_AG_NAME  = "SorghumAvailable_NotCurrentAg_R"

# -------------------------
# OPTIONAL: align outputs to landuse grid (recommended)
# -------------------------
ALIGN_TO_LANDUSE = True


## 2) Validate inputs + set workspace

In [ ]:
arcpy.env.workspace = GDB

landuse_path = os.path.join(GDB, LANDUSE_RASTER_NAME)
if not arcpy.Exists(landuse_path):
    raise RuntimeError(f"Land-use raster not found: {landuse_path}")

lu_r = Raster(landuse_path)

if ALIGN_TO_LANDUSE:
    arcpy.env.snapRaster = landuse_path
    arcpy.env.cellSize   = landuse_path
    arcpy.env.extent     = landuse_path

print("Workspace:", GDB)
print("Land-use raster:", LANDUSE_RASTER_NAME)


## 3) Define CDL/CropScape class codes

In [ ]:
# -------------------------
# CLASS CODES
# -------------------------
# 1) Current agriculture (actively cropped / hay, etc.)
AG_CURRENT_CODES = [
     1,   # Corn
     4,   # Sorghum
     5,   # Soybeans
     6,   # Sunflower
    12,   # Sweet Corn
    13,   # Pop or Orn Corn
    14,   # Mint
    21,   # Barley
    22,   # Durum Wheat
    23,   # Spring Wheat
    24,   # Winter Wheat
    27,   # Rye
    28,   # Oats
    29,   # Millet
    31,   # Canola
    32,   # Flaxseed
    35,   # Mustard
    36,   # Alfalfa
    37,   # Other Hay/Non Alfalfa
    38,   # Camelina
    39,   # Buckwheat
    41,   # Sugarbeets
    42,   # Dry Beans
    43,   # Potatoes
    44,   # Other Crops
    48,   # Watermelons
    53,   # Peas
    58,   # Clover/Wildflowers
    59,   # Sod/Grass Seed
    60,   # Switchgrass
   205,   # Triticale
   229,   # Pumpkins
   246    # Radishes
]

# 2) Convertible land (allowed as sorghum-available)
CONVERTIBLE_CODES = [
    61,   # Fallow/Idle Cropland
   176    # Grassland/Pasture
]

# 3) Hard exclusions (never count as available)
EXCLUDE_CODES = [
    0,           # Background
    111,         # Open Water
    121, 122, 123, 124,  # Developed classes
    131,         # Barren
    141, 142, 143,       # Forest
    152,         # Shrubland
    190, 195     # Wetlands
]

CANDIDATE_CODES = AG_CURRENT_CODES + CONVERTIBLE_CODES


## 4) Helper functions (mask builder + area calculator)

In [ ]:
def code_mask(raster_obj, codes):
    # Boolean raster: True where raster value is in `codes`
    if not codes:
        return raster_obj * 0

    m = (raster_obj == codes[0])
    for c in codes[1:]:
        m = m | (raster_obj == c)
    return m


def raster_area_stats(binary_raster_path, value_to_count=1):
    # Area stats for a binary raster using the raster attribute table (RAT)
    arcpy.management.BuildRasterAttributeTable(binary_raster_path, "Overwrite")

    r = Raster(binary_raster_path)
    cell_area_m2 = r.meanCellWidth * r.meanCellHeight

    total_cells = 0
    with arcpy.da.SearchCursor(binary_raster_path, ["Value", "Count"]) as cur:
        for val, cnt in cur:
            if val == value_to_count and cnt is not None:
                total_cells += cnt

    area_m2 = total_cells * cell_area_m2
    return {
        "cells": total_cells,
        "area_m2": area_m2,
        "area_ha": area_m2 / 10_000.0,
        "area_ac": area_m2 / 4046.8564224,
        "cell_area_m2": cell_area_m2,
    }


def total_area_from_raster(raster_path):
    # Total area for all non-null cells in a raster (all classes)
    arcpy.management.BuildRasterAttributeTable(raster_path, "Overwrite")

    r = Raster(raster_path)
    cell_area_m2 = r.meanCellWidth * r.meanCellHeight

    total_cells = 0
    with arcpy.da.SearchCursor(raster_path, ["Value", "Count"]) as cur:
        for _, cnt in cur:
            if cnt is not None:
                total_cells += cnt

    area_m2 = total_cells * cell_area_m2
    return {
        "cells": total_cells,
        "area_m2": area_m2,
        "area_ha": area_m2 / 10_000.0,
        "area_ac": area_m2 / 4046.8564224,
        "cell_area_m2": cell_area_m2,
    }


## 5) Build rasters: current agriculture + sorghum-available

In [ ]:
current_ag_path = os.path.join(GDB, CURRENT_AG_NAME)

# Raster 1: Current agriculture (1 = current ag; NoData elsewhere)
if not arcpy.Exists(current_ag_path):
    current_ag_mask = code_mask(lu_r, AG_CURRENT_CODES)
    current_ag_r = Con(current_ag_mask, 1)
    current_ag_r.save(current_ag_path)
    print("Saved:", current_ag_path)
else:
    print("Exists:", current_ag_path)


sorghum_avail_path = os.path.join(GDB, SORGHUM_AVAILABLE_NAME)

# Raster 2: Current ag + sorghum-available (1 = candidate; NoData elsewhere)
if not arcpy.Exists(sorghum_avail_path):
    candidate_mask = code_mask(lu_r, CANDIDATE_CODES)
    exclude_mask   = code_mask(lu_r, EXCLUDE_CODES)
    sorghum_avail_mask = candidate_mask & ~exclude_mask

    sorghum_avail_r = Con(sorghum_avail_mask, 1)
    sorghum_avail_r.save(sorghum_avail_path)
    print("Saved:", sorghum_avail_path)
else:
    print("Exists:", sorghum_avail_path)


## 6) Derive sorghum-available land that is not current agriculture

In [ ]:
available_not_ag_path = os.path.join(GDB, AVAILABLE_NOT_AG_NAME)

if not arcpy.Exists(available_not_ag_path):
    current_ag_r = Raster(current_ag_path)
    sorghum_avail_r = Raster(sorghum_avail_path)

    # available == 1 AND CurrentAg is NULL (not currently cropped)
    condition = (sorghum_avail_r == 1) & IsNull(current_ag_r)

    out_r = Con(condition, 1)
    out_r.save(available_not_ag_path)
    print("Saved:", available_not_ag_path)
else:
    print("Exists:", available_not_ag_path)


## 7) Area summary (current ag, available-not-ag, total area, percent)

In [ ]:
stats_current_ag = raster_area_stats(current_ag_path, value_to_count=1)
stats_avail_not_ag = raster_area_stats(available_not_ag_path, value_to_count=1)
stats_total = total_area_from_raster(landuse_path)

pct_current_ag = (stats_current_ag["area_m2"] / stats_total["area_m2"] * 100.0) if stats_total["area_m2"] else 0.0
pct_avail_not_ag = (stats_avail_not_ag["area_m2"] / stats_total["area_m2"] * 100.0) if stats_total["area_m2"] else 0.0

summary = pd.DataFrame([
    {
        "metric": "Current agriculture (Value=1)",
        "cells": stats_current_ag["cells"],
        "area_m2": round(stats_current_ag["area_m2"], 0),
        "area_ha": round(stats_current_ag["area_ha"], 1),
        "area_ac": round(stats_current_ag["area_ac"], 1),
        "pct_of_total": round(pct_current_ag, 2),
    },
    {
        "metric": "Sorghum-available NOT current ag (Value=1)",
        "cells": stats_avail_not_ag["cells"],
        "area_m2": round(stats_avail_not_ag["area_m2"], 0),
        "area_ha": round(stats_avail_not_ag["area_ha"], 1),
        "area_ac": round(stats_avail_not_ag["area_ac"], 1),
        "pct_of_total": round(pct_avail_not_ag, 2),
    },
    {
        "metric": "Total area (all land-use cells)",
        "cells": stats_total["cells"],
        "area_m2": round(stats_total["area_m2"], 0),
        "area_ha": round(stats_total["area_ha"], 1),
        "area_ac": round(stats_total["area_ac"], 1),
        "pct_of_total": 100.00,
    }
])

print(f"Cell area (m²): {stats_total['cell_area_m2']}")
summary


## 8) Cleanup

In [ ]:
arcpy.CheckInExtension("Spatial")
